# Music catalog exploration

In [ ]:
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
import sklearn

SEED = 42

import os
import urllib.request

DATA_URL = 'https://github.com/nthu-cs-hmilab/I2ML-2026/releases/download/data/tracks.csv'
LOCAL_PATH = 'tracks.csv'  # if you downloaded the file from eeclass, put it here or change this path

if not os.path.exists(LOCAL_PATH):
    urllib.request.urlretrieve(DATA_URL, LOCAL_PATH)

df = pd.read_csv(LOCAL_PATH)
df.shape


# HW1: Exploratory Analysis of a Music Catalog
Only the provided catalog is used. Tables and figures are also saved under `output/`.
This notebook supplies reproducible computations and methodological notes; write the final report in your own words.

## Setup
The starter cells above install packages and load the dataset. No GPU is needed.

In [ ]:
import sys, json, hashlib
from pathlib import Path
from IPython.display import display
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor

np.random.seed(SEED)
OUT = Path('output'); OUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_columns', 30)
def table(name, value):
    value.to_csv(OUT / f'{name}.csv')
    display(value)
def figure(name):
    plt.tight_layout(); plt.savefig(OUT / f'{name}.png', dpi=160, bbox_inches='tight'); plt.show()
def rmse(y, pred):
    return float(np.sqrt(mean_squared_error(y, pred)))

raw = df  # Reuse the starter's loaded data; raw is never mutated.
AUDIO = ['danceability', 'energy', 'speechiness', 'acousticness',
         'instrumentalness', 'liveness', 'valence', 'loudness', 'tempo']
CATEGORICAL = ['key', 'mode', 'time_signature']
BASE_INPUTS = [c for c in AUDIO if c != 'energy'] + CATEGORICAL
assert set(AUDIO + BASE_INPUTS + ['track_id', 'track_genre', 'popularity']).issubset(raw.columns)
versions = {k: v for k, v in [('python', sys.version), ('numpy', np.__version__),
    ('pandas', pd.__version__), ('scipy', scipy.__version__), ('sklearn', sklearn.__version__)]}
print('Raw shape:', raw.shape, '\nVersions:', versions)

## 2. Recorded pre-registration
These predictions were transcribed from `the preregistration report`, not revised after observing results:
energy RMSE **0.14** (0–1 units); popularity RMSE **30** (0–100 units).
Expectations: danceability and valence are positively related; loudness and popularity are weakly related.
The feature-estimation accuracy and popularity collection date cannot be verified from this CSV alone.

## 3. Audit raw data, then apply explicit cleaning decisions
Raw data are used only in this audit. Later analyses use one observation per track ID.
Genre memberships are retained separately so multi-genre tracks are not assigned an arbitrary first genre.

In [ ]:
missing = raw.isna().sum().to_frame('missing_rows')
table('audit_missing', missing)
exact_duplicates = int(raw.duplicated().sum())
id_counts = raw.groupby('track_id').size()
unique_per_id = raw.groupby('track_id').nunique(dropna=False)
conflict_counts = unique_per_id.gt(1).sum().to_frame('IDs_with_multiple_values')
table('audit_id_conflicts', conflict_counts)
conflict_ids = unique_per_id.index[unique_per_id['popularity'] > 1]
table('audit_conflict_examples', raw.loc[raw.track_id.isin(conflict_ids),
      ['track_id', 'track_genre', 'popularity']].head(20))

bounded = ['danceability', 'energy', 'speechiness', 'acousticness',
           'instrumentalness', 'liveness', 'valence']
checks = {c: ~raw[c].between(0, 1) for c in bounded}
checks.update({'popularity': ~raw.popularity.between(0, 100),
    'duration_ms': raw.duration_ms.le(0), 'tempo': raw.tempo.le(0),
    'key': ~raw.key.isin(range(12)), 'mode': ~raw['mode'].isin([0, 1]),
    'time_signature': ~raw.time_signature.isin([1, 2, 3, 4, 5]),
    'nonfinite': ~np.isfinite(raw[AUDIO + CATEGORICAL + ['popularity', 'duration_ms']]).all(axis=1)})
invalid_flags = pd.DataFrame(checks)
invalid = invalid_flags.any(axis=1)
table('audit_invalid', invalid_flags.sum().to_frame('flagged_rows'))
table('audit_invalid_examples', raw.loc[invalid].head(10))
table('audit_numeric_quantiles', raw[AUDIO + ['duration_ms', 'popularity']].quantile([0, .01, .5, .99, 1]).T)

# Four prioritized findings, with operational decisions and their costs.
audit_decisions = pd.DataFrame([
    [1, 'Repeated track identities', f'{int((id_counts > 1).sum())} repeated IDs; {exact_duplicates} exact duplicate rows',
     'Repeated tracks overweight geometry and can cross random row splits.',
     'Remove exact duplicates, use one row per ID, preserve all genre memberships; split by ID.',
     'Listing frequency is discarded; findings describe unique tracks.'],
    [2, 'Conflicting popularity for an ID', f'{len(conflict_ids)} IDs',
     'The same track has ambiguous regression labels.',
     'Exclude these IDs from the common cleaned analysis dataset rather than invent a target.',
     'Valid audio observations are also lost; results apply to the consistent-label subset.'],
    [3, 'Zero or invalid measurements', f'{int(invalid.sum())} rows with at least one flagged field',
     'Zero tempo/duration or unsupported meter can distort distances and models.',
     'Remove flagged listings; retain legitimate extremes and positive loudness.',
     'Unusual music or unrecognized meter may be removed; zeros may mean unknown rather than error.'],
    [4, 'Missing descriptive metadata', f'{int(raw[["artists", "album_name", "track_name"]].isna().any(axis=1).sum())} rows',
     'Missing names affect interpretation but are not regression inputs.',
     'Fill missing descriptive strings with <unknown>; do not discard valid audio.',
     'The original identity description remains unknown.'],
], columns=['priority', 'finding', 'evidence', 'impact', 'decision', 'trade_off']).set_index('priority')
table('audit_decisions', audit_decisions)
print('Ranking rationale: repeated identities affect every analysis and split; conflicting popularity affects fewer IDs and primarily the labels.')

In [ ]:
dedup = raw.drop_duplicates().copy()
valid_listings = dedup.loc[~invalid.loc[dedup.index] & ~dedup.track_id.isin(conflict_ids)].copy()
for c in ['artists', 'album_name', 'track_name']:
    valid_listings[c] = valid_listings[c].fillna('<unknown>')
memberships = valid_listings[['track_id', 'track_genre']].drop_duplicates()
clean = valid_listings.drop_duplicates('track_id').drop(columns='track_genre').reset_index(drop=True)
# Genre strings are a deterministic set, never a choice based on target values.
genre_sets = memberships.groupby('track_id').track_genre.agg(lambda x: '|'.join(sorted(x.unique())))
clean['genre_memberships'] = clean.track_id.map(genre_sets)
clean['genre_count'] = clean.track_id.map(memberships.groupby('track_id').size())
assert clean.track_id.is_unique and np.isfinite(clean[AUDIO]).all().all()
assert not clean.track_id.isin(conflict_ids).any()
clean.to_csv(OUT / 'clean_tracks.csv', index=False)
memberships.to_csv(OUT / 'clean_genre_memberships.csv', index=False)
table('cleaning_counts', pd.DataFrame({'rows': [len(raw), len(dedup), len(valid_listings), len(clean)]},
    index=['raw listings', 'distinct listings', 'valid consistent listings', 'unique tracks']))
correlations = clean[['danceability', 'valence', 'loudness', 'popularity']].corr()
table('preregistration_correlations', correlations)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (x, y) in zip(axes, [('danceability', 'valence'), ('loudness', 'popularity')]):
    im = ax.hexbin(clean[x], clean[y], gridsize=45, mincnt=1, bins='log')
    ax.set(xlabel=x, ylabel=y, title=f'Pearson r = {clean[x].corr(clean[y]):.3f}')
    fig.colorbar(im, ax=ax, label='log10 count')
figure('preregistration_relationships')

## 4. PCA: continuous audio geometry
The nine continuous audio descriptors include energy; popularity and duration are excluded.
Key, mode, and meter are categories, so they are not treated as Euclidean coordinates here.
The primary sufficiency criterion is **90% cumulative explained variance**, with 80% and 95% sensitivity checks.
PCA maximizes projected variance: for a unit vector $w$, $\mathrm{Var}(Xw)=w^T S w$.
Scaling replaces $S$ with the correlation matrix. Unscaled PCA therefore gives larger-unit features more influence.

In [ ]:
audio_scaler = StandardScaler()
Z = audio_scaler.fit_transform(clean[AUDIO])
pca = PCA(svd_solver='full', random_state=SEED).fit(Z)
raw_pca = PCA(svd_solver='full', random_state=SEED).fit(clean[AUDIO])
scores = pca.transform(Z)
variance = pd.DataFrame({'standardized_variance': pca.explained_variance_ratio_,
    'raw_variance': raw_pca.explained_variance_ratio_,
    'standardized_cumulative': np.cumsum(pca.explained_variance_ratio_),
    'raw_cumulative': np.cumsum(raw_pca.explained_variance_ratio_)}, index=np.arange(1, len(AUDIO)+1))
table('pca_variance', variance)
table('pca_dimensions', pd.DataFrame({name: [np.searchsorted(np.cumsum(model.explained_variance_ratio_), t)+1
    for t in [.8, .9, .95]] for name, model in [('standardized', pca), ('raw', raw_pca)]}, index=[.8, .9, .95]))
loadings = pd.DataFrame(pca.components_.T, index=AUDIO, columns=[f'PC{i+1}' for i in range(len(AUDIO))])
table('pca_loadings', loadings.iloc[:, :3])
table('pca_raw_loadings', pd.DataFrame(raw_pca.components_[:3].T, index=AUDIO, columns=['PC1', 'PC2', 'PC3']))
cov_raw = np.cov(clean[AUDIO].to_numpy(), rowvar=False)
cov_std = np.cov(Z, rowvar=False)
variance_check = pd.DataFrame({'raw_feature_variance': np.diag(cov_raw),
    'raw_variance_share': np.diag(cov_raw)/np.trace(cov_raw), 'scaled_feature_variance': np.diag(cov_std)}, index=AUDIO)
table('pca_scaling_evidence', variance_check)
table('pca_projected_variance_check', pd.DataFrame([
    {'scaling': name, 'PC1_projected_variance': float(model.components_[0] @ cov @ model.components_[0]),
     'PC1_eigenvalue': model.explained_variance_[0], 'total_variance': np.trace(cov)}
    for name, model, cov in [('raw', raw_pca, cov_raw), ('standardized', pca, cov_std)]]))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
variance[['standardized_cumulative', 'raw_cumulative']].plot(marker='o', ax=axes[0])
axes[0].axhline(.9, color='gray', linestyle='--'); axes[0].set(xlabel='Number of PCs', ylabel='Cumulative variance')
im = axes[1].imshow(loadings.iloc[:, :3], cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
axes[1].set(yticks=range(len(AUDIO)), yticklabels=AUDIO, xticks=range(3), xticklabels=['PC1', 'PC2', 'PC3'])
fig.colorbar(im, ax=axes[1], label='Eigenvector weight'); figure('pca_comparison')
for pc in ['PC1', 'PC2', 'PC3']:
    weights = loadings[pc]
    print(pc, 'largest positive weights:', weights.nlargest(3).round(3).to_dict(),
          'largest negative weights:', weights.nsmallest(3).round(3).to_dict())

## 5.1 Hierarchical clustering of genres
Each genre profile is the mean of its distinct cleaned tracks, standardized using the track-level scaler.
Compare all four combinations of Euclidean/Manhattan distance and average/complete linkage.
At a common cut of six clusters, compare memberships, silhouette in each method's own metric,
and cophenetic correlation. Distances' absolute scales cannot be compared directly.

In [ ]:
genre_data = memberships.merge(clean[['track_id'] + AUDIO], on='track_id', validate='many_to_one')
genre_profiles = genre_data.groupby('track_genre')[AUDIO].mean()
genre_Z = audio_scaler.transform(genre_profiles)
hierarchy_labels, hierarchy_rows = {}, []
fig, axes = plt.subplots(2, 2, figsize=(22, 14))
for ax, (metric, method) in zip(axes.flat, [(m, l) for m in ['euclidean', 'cityblock'] for l in ['average', 'complete']]):
    distance = pdist(genre_Z, metric=metric)
    tree = linkage(distance, method=method)
    labels = fcluster(tree, t=6, criterion='maxclust')
    name = f'{metric}_{method}'; hierarchy_labels[name] = labels
    hierarchy_rows.append({'method': name, 'clusters': len(np.unique(labels)),
        'silhouette': silhouette_score(squareform(distance), labels, metric='precomputed'),
        'cophenetic_correlation': cophenet(tree, distance)[0]})
    dendrogram(tree, labels=genre_profiles.index.tolist(), leaf_rotation=90, leaf_font_size=5, ax=ax)
    ax.set(title=name, ylabel='Dissimilarity')
figure('genre_dendrograms')
hierarchy_metrics = pd.DataFrame(hierarchy_rows).set_index('method')
table('genre_hierarchy_metrics', hierarchy_metrics)
table('genre_hierarchy_memberships', pd.DataFrame(hierarchy_labels, index=genre_profiles.index))
table('genre_hierarchy_agreement', pd.DataFrame({a: {b: adjusted_rand_score(x, y)
    for b, y in hierarchy_labels.items()} for a, x in hierarchy_labels.items()}))
preferred_hierarchy = hierarchy_metrics.silhouette.idxmax()
table('genre_preferred_cluster_profiles', genre_profiles.assign(cluster=hierarchy_labels[preferred_hierarchy]).groupby('cluster').mean())
print('Candidate supported by highest silhouette at this fixed cut:', preferred_hierarchy,
      '; also inspect its profiles and cophenetic correlation before a musical interpretation.')

## 5.2 Individual-track clustering
For model selection, uniformly sample up to 12,000 unique tracks without replacement with the fixed seed.
Evaluate inertia and silhouette on the same 2,000 sampled tracks for k=2…10.
Choose the silhouette maximum; inspect the inertia curve as a second criterion rather than assume an elbow exists.
Fit the selected k on all cleaned tracks. Standardization uses the cleaned tracks because this is descriptive analysis.

In [ ]:
rng = np.random.default_rng(SEED)
selection_idx = rng.choice(len(clean), size=min(12000, len(clean)), replace=False)
selection_Z = Z[selection_idx]
eval_idx = rng.choice(len(selection_idx), size=min(2000, len(selection_idx)), replace=False)
k_rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(selection_Z)
    k_rows.append({'k': k, 'inertia': km.inertia_,
        'silhouette': silhouette_score(selection_Z[eval_idx], km.labels_[eval_idx])})
k_scores = pd.DataFrame(k_rows).set_index('k')
k_scores['relative_inertia_drop'] = -k_scores.inertia.pct_change()
table('kmeans_selection', k_scores)
BEST_K = int(k_scores.silhouette.idxmax())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
k_scores.inertia.plot(marker='o', ax=axes[0]); axes[0].set(ylabel='Inertia', xlabel='k')
k_scores.silhouette.plot(marker='o', ax=axes[1]); axes[1].set(ylabel='Silhouette', xlabel='k')
for ax in axes: ax.axvline(BEST_K, color='gray', linestyle='--')
figure('kmeans_selection')
track_kmeans = KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED).fit(Z)
track_labels = track_kmeans.labels_
cluster_means = clean[AUDIO].assign(cluster=track_labels).groupby('cluster').mean()
table('track_cluster_means', cluster_means)
table('track_cluster_sizes', pd.Series(track_labels).value_counts().sort_index().to_frame('tracks'))
cluster_zmeans = pd.DataFrame(audio_scaler.transform(cluster_means), index=cluster_means.index, columns=AUDIO)
table('track_cluster_standardized_means', cluster_zmeans)
fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(cluster_zmeans, cmap='coolwarm', vmin=-2, vmax=2, aspect='auto')
ax.set(xticks=range(len(AUDIO)), xticklabels=AUDIO, yticks=range(BEST_K), yticklabels=range(BEST_K), ylabel='Cluster')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right'); fig.colorbar(im, label='Mean relative to all tracks (SD)')
figure('track_cluster_profiles')
for c, row in cluster_zmeans.iterrows():
    print('Cluster', c, 'higher:', row.nlargest(3).round(2).to_dict(), 'lower:', row.nsmallest(3).round(2).to_dict())

## 5.3 Evidence beyond algorithm-imposed groups
Evidence includes PC1/PC2 projection, adjusted Rand agreement under independent sampling and initialization,
genre-balanced sampling, and a feature-shuffled null preserving each feature's marginal distribution.
The null asks about cross-feature structure; it is not a formal significance test.
High stability or separation alone does not establish discrete natural genres.

In [ ]:
plot_idx = rng.choice(len(clean), size=min(6000, len(clean)), replace=False)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.scatter(scores[plot_idx, 0], scores[plot_idx, 1], c=track_labels[plot_idx], cmap='tab10', s=5, alpha=.35)
ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
fig.colorbar(im, label='Cluster'); figure('clusters_in_pca')
stability_rows = []
for repetition in range(3):
    subset = rng.choice(len(clean), size=min(12000, len(clean)), replace=False)
    alt = KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED).fit(Z[subset])
    stability_rows.append({'experiment': f'independent sample {repetition+1}',
        'ARI_vs_full': adjusted_rand_score(track_labels, alt.predict(Z))})
# Different initialization supplied explicitly by seeded RNG, while random_state stays SEED.
init_centers = Z[rng.choice(len(clean), BEST_K, replace=False)]
alt = KMeans(n_clusters=BEST_K, init=init_centers, n_init=1, random_state=SEED).fit(Z)
stability_rows.append({'experiment': 'different initial centers', 'ARI_vs_full': adjusted_rand_score(track_labels, alt.labels_)})
# Select equal listing counts per genre, then collapse any shared IDs before fitting.
balanced_ids = (memberships.groupby('track_genre', group_keys=False)
    .apply(lambda g: g.sample(n=min(100, len(g)), random_state=SEED), include_groups=False).track_id.unique())
balanced_idx = np.flatnonzero(clean.track_id.isin(balanced_ids))
balanced_model = KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED).fit(Z[balanced_idx])
stability_rows.append({'experiment': 'genre-balanced memberships, unique IDs',
    'ARI_vs_full': adjusted_rand_score(track_labels, balanced_model.predict(Z))})
table('cluster_stability', pd.DataFrame(stability_rows).set_index('experiment'))
null_Z = selection_Z.copy()
for j in range(null_Z.shape[1]): null_Z[:, j] = rng.permutation(null_Z[:, j])
null_model = KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED).fit(null_Z)
observed_model = KMeans(n_clusters=BEST_K, n_init=10, random_state=SEED).fit(selection_Z)
table('cluster_null_comparison', pd.DataFrame({'silhouette': [
    silhouette_score(selection_Z[eval_idx], observed_model.labels_[eval_idx]),
    silhouette_score(null_Z[eval_idx], null_model.labels_[eval_idx])]}, index=['observed', 'independent feature permutations']))
cluster_genres = memberships.merge(clean[['track_id']].assign(cluster=track_labels), on='track_id')
table('cluster_genre_counts', pd.crosstab(cluster_genres.cluster, cluster_genres.track_genre))
print('Balanced clustering sample:', len(balanced_idx), 'unique tracks. Genre labels were never clustering inputs.')

## 6.1–6.2 Leakage-safe regression and recorded predictions
Use a common 60/20/20 train/validation/test partition grouped by track ID. Preprocessing is fitted only on training data.
Q1/Q2 inputs are exactly the allowed audio features; energy is added only for popularity.
Musical categories are one-hot encoded with a dropped reference; duration, metadata, and genre are excluded.
Compare tasks using baseline-relative RMSE and R², since their raw units differ.
Validation is used for regularization choices; test data are reserved for final evaluation.

In [ ]:
split = GroupShuffleSplit(n_splits=1, test_size=.2, random_state=SEED)
development_idx, test_idx = next(split.split(clean, groups=clean.track_id))
inner = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED)
train_local, val_local = next(inner.split(clean.iloc[development_idx], groups=clean.iloc[development_idx].track_id))
train_idx, val_idx = development_idx[train_local], development_idx[val_local]
train, validation, test = [clean.iloc[i].copy() for i in [train_idx, val_idx, test_idx]]
id_sets = [set(x.track_id) for x in [train, validation, test]]
assert all(not id_sets[i] & id_sets[j] for i in range(3) for j in range(i+1, 3))
partition = pd.Series('train', index=clean.index)
partition.iloc[val_idx] = 'validation'; partition.iloc[test_idx] = 'test'
clean[['track_id']].assign(partition=partition).to_csv(OUT / 'regression_split.csv', index=False)
table('regression_partition_sizes', partition.value_counts().to_frame('unique_tracks'))

def preprocessing(inputs):
    continuous = [c for c in inputs if c not in CATEGORICAL]
    categories = [c for c in inputs if c in CATEGORICAL]
    return ColumnTransformer([('continuous', StandardScaler(), continuous),
        ('categories', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), categories)])

regression_rows, regressors = [], {}
predicted_errors = {'energy': .14, 'popularity': 30.0}
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, target in zip(axes, ['energy', 'popularity']):
    inputs = BASE_INPUTS + (['energy'] if target == 'popularity' else [])
    assert target not in inputs and set(inputs).issubset(set(BASE_INPUTS + ['energy']))
    model = make_pipeline(preprocessing(inputs), LinearRegression()).fit(train[inputs], train[target])
    baseline = DummyRegressor(strategy='mean').fit(train[inputs], train[target])
    pred = model.predict(test[inputs]); base_pred = baseline.predict(test[inputs])
    regressors[target] = (model, inputs)
    baseline_rmse = rmse(test[target], base_pred)
    regression_rows.append({'target': target, 'preregistered_RMSE': predicted_errors[target],
        'linear_test_RMSE': rmse(test[target], pred), 'mean_baseline_test_RMSE': baseline_rmse,
        'relative_RMSE_reduction': 1-rmse(test[target], pred)/baseline_rmse,
        'test_R2': r2_score(test[target], pred), 'training_mean': float(train[target].mean())})
    ax.hexbin(test[target], pred, gridsize=45, mincnt=1, bins='log')
    lo, hi = test[target].min(), test[target].max(); ax.plot([lo, hi], [lo, hi], 'r--')
    ax.set(xlabel=f'Observed {target}', ylabel='Predicted', title=target)
    test[['track_id', target]].assign(prediction=pred, baseline=base_pred).to_csv(OUT / f'{target}_predictions.csv', index=False)
table('regression_results', pd.DataFrame(regression_rows).set_index('target'))
figure('regression_predictions')
target_correlations = train[AUDIO + ['popularity']].corr()[['energy', 'popularity']]
table('training_target_correlations', target_correlations)
target_correlations.plot.bar(figsize=(11, 4)); plt.ylabel('Pearson correlation (training tracks)'); figure('target_correlations')

## 6.3 One additional non-audio variable: genre membership
Use all retained memberships as multi-hot indicators (no arbitrary primary genre).
Genre-only ridge and audio-plus-genre ridge select strength on validation and evaluate the same held-out tracks.
Genre means are measured on training tracks only. A track can contribute to multiple genre means.
Genre effects may reflect collection, exposure, or label assignment rather than causal audio properties.

In [ ]:
genre_matrix = pd.crosstab(memberships.track_id, memberships.track_genre).clip(upper=1).reindex(clean.track_id).to_numpy(dtype=float)
Gtrain, Gval, Gtest = [genre_matrix[i] for i in [train_idx, val_idx, test_idx]]
genre_training = memberships.merge(train[['track_id', 'popularity']], on='track_id')
genre_summary = genre_training.groupby('track_genre').popularity.agg(['count', 'mean', 'std']).sort_values('mean')
table('training_genre_popularity', genre_summary)
pd.concat([genre_summary.head(10), genre_summary.tail(10)])['mean'].plot.barh(figsize=(8, 6))
plt.xlabel('Training popularity mean'); figure('genre_popularity')
audio_model, pop_inputs = regressors['popularity']
pop_transform = preprocessing(pop_inputs).fit(train[pop_inputs])
P_train, P_val, P_test = [pop_transform.transform(x[pop_inputs]) for x in [train, validation, test]]
extra_rows = []
for name, a, b, c in [('genre only', Gtrain, Gval, Gtest),
    ('audio + genre', np.c_[P_train, Gtrain], np.c_[P_val, Gval], np.c_[P_test, Gtest])]:
    candidates = []
    for alpha in [.01, 1, 100, 10000]:
        model = Ridge(alpha=alpha).fit(a, train.popularity)
        candidates.append((rmse(validation.popularity, model.predict(b)), alpha, model))
    val_error, alpha, selected = min(candidates, key=lambda item: item[0])
    extra_rows.append({'inputs': name, 'alpha': alpha, 'validation_RMSE': val_error,
                       'test_RMSE': rmse(test.popularity, selected.predict(c))})
extra_rows.append({'inputs': 'audio only (Q1 OLS)', 'alpha': 0, 'validation_RMSE':
    rmse(validation.popularity, audio_model.predict(validation[pop_inputs])),
    'test_RMSE': rmse(test.popularity, audio_model.predict(test[pop_inputs]))})
table('additional_genre_information', pd.DataFrame(extra_rows).set_index('inputs'))

## 6.4 Energy regularization
Select a correlated three-feature set using **training inputs only**: find the predictor with the most
correlations of magnitude at least 0.4, then take its two most correlated neighbors.
These are the strongest available relationships, not assumed near-perfect collinearity; the neighbors need not
correlate with one another. Display their actual pairwise correlations. Standardize before fitting ridge over 10⁻⁴…10⁶.
Select alpha on validation, then report test RMSE across the prespecified path as a diagnostic, without choosing from test results.
Also run a controlled stress experiment: 200 training tracks with degree-five polynomial bases of the same predictors.
This experiment always runs so that an improvement is not manufactured by repeatedly inspecting the test set.
RMSE reductions of at most 10⁻⁶ are reported as numerically negligible rather than meaningful improvements.

In [ ]:
continuous_inputs = [c for c in BASE_INPUTS if c not in CATEGORICAL]
input_corr = train[continuous_inputs].corr()
abs_corr = input_corr.abs().where(~np.eye(len(input_corr), dtype=bool), 0)
hub = abs_corr.ge(.4).sum().idxmax()
REG_FEATURES = [hub] + abs_corr.loc[hub].drop(hub).nlargest(2).index.tolist()
table('regularization_correlated_features', input_corr.loc[REG_FEATURES, REG_FEATURES])
pair_rows = [{'feature_1': a, 'feature_2': b, 'correlation': input_corr.loc[a, b]}
    for i, a in enumerate(continuous_inputs) for b in continuous_inputs[i+1:]]
table('regularization_pairwise_correlations', pd.DataFrame(pair_rows).assign(
    abs_correlation=lambda x: x.correlation.abs()).sort_values('abs_correlation', ascending=False))
ALPHAS = np.logspace(-4, 6, 16)
def ridge_experiment(name, training, polynomial=False):
    steps = [StandardScaler()]
    if polynomial: steps += [PolynomialFeatures(degree=5, include_bias=False), StandardScaler()]
    transform = make_pipeline(*steps).fit(training[REG_FEATURES])
    a, b, c = [transform.transform(x[REG_FEATURES]) for x in [training, validation, test]]
    models, rows = [], []
    ols = LinearRegression().fit(a, training.energy)
    ols_val = rmse(validation.energy, ols.predict(b))
    for alpha in ALPHAS:
        model = Ridge(alpha=alpha).fit(a, training.energy)
        models.append(model)
        rows.append({'alpha': alpha, 'train_RMSE': rmse(training.energy, model.predict(a)),
                     'validation_RMSE': rmse(validation.energy, model.predict(b)),
                     'coefficient_L2': np.linalg.norm(model.coef_)})
    results = pd.DataFrame(rows).set_index('alpha')
    best_pos = int(np.argmin(results.validation_RMSE.to_numpy()))  # Freeze before test evaluation.
    results['test_RMSE'] = [rmse(test.energy, m.predict(c)) for m in models]
    ols_test = rmse(test.energy, ols.predict(c))
    table(f'{name}_ridge_path', results)
    names = transform.get_feature_names_out(REG_FEATURES)
    coef = pd.DataFrame([m.coef_ for m in models], index=ALPHAS, columns=names)
    coef.to_csv(OUT / f'{name}_coefficients.csv')
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    results[['validation_RMSE', 'test_RMSE']].plot(logx=True, ax=axes[0])
    axes[0].axhline(ols_test, color='gray', linestyle='--', label='OLS test')
    axes[0].axvline(ALPHAS[best_pos], color='black', linestyle=':'); axes[0].set(ylabel='Energy RMSE', xlabel='alpha'); axes[0].legend()
    if polynomial:
        results.coefficient_L2.plot(logx=True, ax=axes[1]); axes[1].set(ylabel='Coefficient L2 norm')
    else:
        coef.plot(logx=True, ax=axes[1]); axes[1].set(ylabel='Standardized coefficient')
    axes[1].set(xlabel='alpha'); figure(f'{name}_regularization')
    return {'experiment': name, 'training_tracks': len(training), 'basis_features': a.shape[1],
        'selected_alpha': ALPHAS[best_pos], 'OLS_validation_RMSE': ols_val, 'ridge_validation_RMSE': results.validation_RMSE.iloc[best_pos],
        'OLS_test_RMSE': ols_test, 'selected_ridge_test_RMSE': results.test_RMSE.iloc[best_pos],
        'ridge_test_improved': bool(ols_test-results.test_RMSE.iloc[best_pos] > 1e-6)}

small_train = train.sample(n=min(200, len(train)), random_state=SEED)
reg_summaries = pd.DataFrame([ridge_experiment('full_training', train),
                             ridge_experiment('small_polynomial', small_train, polynomial=True)]).set_index('experiment')
table('regularization_summary', reg_summaries)
for name, row in reg_summaries.iterrows():
    print(name, 'validation-selected ridge improves test RMSE:', row.ridge_test_improved,
          '; change vs OLS:', row.selected_ridge_test_RMSE-row.OLS_test_RMSE)

## Reproducibility checks and report handoff
The following assertions verify track identity separation, target exclusion, cleaning, and PCA variance calculations.
Use the generated tables/figures to write your own ranking, PC interpretations, cluster descriptions, and conclusions.
Limitations to assess include cleaning selection, multi-genre membership, missing collection time, and the difference
between descriptive correlation and causal or population claims. Review the AI-use log and record your own checks.

In [ ]:
assert sum(map(len, [train, validation, test])) == len(clean)
assert all(not id_sets[i] & id_sets[j] for i in range(3) for j in range(i+1, 3))
assert all(target not in inputs for target, (_, inputs) in regressors.items())
assert clean.track_id.is_unique
assert np.isclose(pca.explained_variance_ratio_.sum(), 1)
assert np.isclose(raw_pca.explained_variance_ratio_.sum(), 1)
assert np.allclose(pca.explained_variance_, np.linalg.eigvalsh(cov_std)[::-1])
manifest = {'seed': SEED, 'dataset_sha256': hashlib.sha256(Path(LOCAL_PATH).read_bytes()).hexdigest(),
    'raw_rows': len(raw), 'clean_unique_tracks': len(clean), 'audio_geometry_features': AUDIO,
    'selected_k': BEST_K, 'regularization_features': REG_FEATURES, 'versions': versions,
    'cluster_selection_tracks': len(selection_idx), 'silhouette_evaluation_tracks': len(eval_idx)}
(OUT / 'reproducibility_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Checks passed. Outputs saved in', OUT.resolve())